In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense

In [6]:
data = pd.read_csv("/content/Amazon_Reviews.csv", sep=',', on_bad_lines='skip', engine='python')

data = data[['Rating', 'Review Text']].dropna()

In [7]:
def get_sentiment(rating):
    rating = int(rating.split()[1])
    if rating >= 4:
        return 1
    elif rating <= 2:
        return 0
    else:
        return None

data['sentiment'] = data['Rating'].apply(get_sentiment)
data = data.dropna()

In [8]:
vocab_size = 10000
max_len = 200

texts = data['Review Text'].astype(str)
labels = data['sentiment'].values

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded = pad_sequences(sequences, maxlen=max_len)

X_train, X_test, y_train, y_test = train_test_split(
    padded, labels, test_size=0.2, random_state=42
)

In [11]:
from tensorflow.keras.layers import Dropout

model = Sequential()
model.add(Embedding(vocab_size, 128, input_length=max_len))
model.add(GRU(64))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

In [12]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=5, batch_size=64)

Epoch 1/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.8787 - loss: 0.2907
Epoch 2/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9517 - loss: 0.1335
Epoch 3/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9680 - loss: 0.0947
Epoch 4/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9743 - loss: 0.0714
Epoch 5/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9696 - loss: 0.0875


In [13]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9321 - loss: 0.1827
Test Accuracy: 0.9320773482322693
